# Sparse CNN Large Run on Google Colab

This notebook runs the sparse-input CNN risk-map experiment on Google Colab.

Goal: use first-pass sparse wafer observations to predict defect-risk scores over unmeasured valid dies. Dense WM-811K maps are used only as offline training/evaluation labels, not as inference inputs.

Before running:

1. In Colab, select `Runtime > Change runtime type > GPU`.
2. Put `LSWMD.pkl` in Google Drive, for example: `/MyDrive/wafer_project_data/LSWMD.pkl`.
3. Either clone the GitHub repo or copy this project folder to Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure Paths

Choose one project setup mode:

- `use_git_clone = True`: clone from GitHub.
- `use_git_clone = False`: use a project folder already copied to Drive.

If the GitHub repo is still private/empty, use the Drive folder mode.

In [ ]:
from pathlib import Path
import os

use_git_clone = False

# If using GitHub, set this to your repository URL.
GITHUB_REPO_URL = 'https://github.com/JUYONG-1010/wafer-risk-followup-sampling.git'

# If using Drive folder mode, copy the local project folder to this path first.
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/wafer-risk-followup-sampling')

# Alternative: upload the prepared zip and let this notebook extract it.
DRIVE_PROJECT_ZIP = Path('/content/drive/MyDrive/wafer-risk-followup-sampling-colab.zip')

# Put the original WM-811K pickle here in Google Drive.
DRIVE_LSWMD_PATH = Path('/content/drive/MyDrive/wafer_project_data/LSWMD.pkl')

WORK_DIR = Path('/content/wafer-risk-followup-sampling')
RESULTS_DIR = Path('/content/drive/MyDrive/wafer_project_results/sparse_cnn_colab_large')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('WORK_DIR:', WORK_DIR)
print('RESULTS_DIR:', RESULTS_DIR)
print('DRIVE_LSWMD_PATH exists:', DRIVE_LSWMD_PATH.exists())

## 3. Get Project Code

In [ ]:
import shutil
import subprocess

if WORK_DIR.exists():
    print('Project already exists at', WORK_DIR)
else:
    if use_git_clone:
        subprocess.run(['git', 'clone', GITHUB_REPO_URL, str(WORK_DIR)], check=True)
    else:
        if DRIVE_PROJECT_DIR.exists():
            shutil.copytree(DRIVE_PROJECT_DIR, WORK_DIR)
        elif DRIVE_PROJECT_ZIP.exists():
            WORK_DIR.mkdir(parents=True, exist_ok=True)
            shutil.unpack_archive(str(DRIVE_PROJECT_ZIP), str(WORK_DIR))
        else:
            raise FileNotFoundError(f'Neither Drive project folder nor project zip was found: {DRIVE_PROJECT_DIR}, {DRIVE_PROJECT_ZIP}')



# Repair archives created on Windows if backslashes were preserved as literal
# filename characters after extraction on Linux/Colab.
moved = 0
for p in list(WORK_DIR.iterdir()):
    if '\\' in p.name:
        dst = WORK_DIR.joinpath(*p.name.split('\\'))
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            if dst.is_dir():
                shutil.rmtree(dst)
            else:
                dst.unlink()
        shutil.move(str(p), str(dst))
        moved += 1
if moved:
    print('repaired flat Windows zip paths:', moved)

os.chdir(WORK_DIR)
print('cwd:', Path.cwd())
print('top-level files:', sorted([p.name for p in WORK_DIR.iterdir()])[:20])

## 4. Install Dependencies

Colab usually already has a CUDA-enabled PyTorch build when GPU runtime is selected. Do not install `requirements-cnn.txt` blindly because it pins the CPU PyTorch wheel. This cell installs only the non-torch project dependencies, then checks PyTorch/GPU.

In [ ]:
import sys
import subprocess

base_packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-image', 'scikit-learn']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *base_packages], check=True)

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU is not available. Use Runtime > Change runtime type > GPU.')

## 5. Place Dataset

The project scripts expect the source pickle at `LSWMD.pkl/LSWMD.pkl`. This cell copies the Drive pickle into that expected local path.

In [ ]:
from pathlib import Path
import shutil

if not DRIVE_LSWMD_PATH.exists():
    raise FileNotFoundError(f'LSWMD pickle not found in Drive: {DRIVE_LSWMD_PATH}')

local_data_dir = WORK_DIR / 'LSWMD.pkl'
local_data_dir.mkdir(parents=True, exist_ok=True)
local_pickle = local_data_dir / 'LSWMD.pkl'

if not local_pickle.exists():
    shutil.copy2(DRIVE_LSWMD_PATH, local_pickle)

print('local pickle:', local_pickle)
print('size MB:', round(local_pickle.stat().st_size / 1024 / 1024, 2))

## 6. Prepare Processed Dataset

The CNN script needs `data/processed/subsets/patterned_subset.pkl`. If it is missing, run the extraction script.

In [ ]:
from pathlib import Path
import subprocess
import sys

patterned_subset = WORK_DIR / 'data/processed/subsets/patterned_subset.pkl'
if patterned_subset.exists():
    print('Found:', patterned_subset)
else:
    print('Missing patterned subset. Running extraction...')
    subprocess.run([sys.executable, 'experiments/01_extract_labeled_subset.py'], check=True)
    print('Created:', patterned_subset, patterned_subset.exists())

## 7. Run Larger Sparse CNN Experiment

Recommended Colab starting point:

- `max_train_wafers`: 5000
- `max_test_wafers`: 800
- `epochs`: 15
- `patience`: 4

If Colab disconnects or runtime is limited, reduce train wafers or epochs.

In [ ]:
import subprocess
import sys
from pathlib import Path

out_dir = WORK_DIR / 'data/processed/sparse_cnn_risk_map_colab_large'
fig_dir = WORK_DIR / 'outputs/figures/58_sparse_cnn_risk_map_colab_large'
model_path = WORK_DIR / 'models/sparse_cnn_risk_map_colab_large.pt'

cmd = [
    sys.executable, 'experiments/68_train_sparse_cnn_risk_map.py',
    '--epochs', '15',
    '--patience', '4',
    '--max-train-wafers', '5000',
    '--max-test-wafers', '800',
    '--densities', '0.01', '0.03', '0.05', '0.10',
    '--hidden-channels', '48',
    '--threads', '4',
    '--out-dir', str(out_dir),
    '--fig-dir', str(fig_dir),
    '--model-path', str(model_path),
]

# Do not pass --cpu-only. The script will use CUDA if available.
print('Running:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

print('out_dir:', out_dir)
print('fig_dir:', fig_dir)
print('model_path:', model_path)

## 7B. Run Optimized Batched Sparse CNN Experiment

This version uses cached masks and padded mini-batches. Use it before attempting larger CNN runs.

It also supports coordinate ablation with `--drop-coordinate-channels`.


In [ ]:
import subprocess
import sys
from pathlib import Path

batched_out_dir = WORK_DIR / 'data/processed/sparse_cnn_risk_map_colab_batched'
batched_fig_dir = WORK_DIR / 'outputs/figures/71_sparse_cnn_risk_map_colab_batched'
batched_model_path = WORK_DIR / 'models/sparse_cnn_risk_map_colab_batched.pt'

cmd = [
    sys.executable, 'experiments/71_train_sparse_cnn_risk_map_batched.py',
    '--epochs', '8',
    '--patience', '3',
    '--max-train-wafers', '2500',
    '--max-test-wafers', '500',
    '--densities', '0.01', '0.03', '0.05', '0.10',
    '--hidden-channels', '48',
    '--batch-size', '16',
    '--num-workers', '0',
    '--threads', '4',
    '--out-dir', str(batched_out_dir),
    '--fig-dir', str(batched_fig_dir),
    '--model-path', str(batched_model_path),
    '--use-amp',
]
print('Running optimized batched CNN:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('batched_out_dir:', batched_out_dir)
print('batched_fig_dir:', batched_fig_dir)
print('batched_model_path:', batched_model_path)


## 8. Generate Visual JPG Examples

In [ ]:
import subprocess
import sys

visual_dir = WORK_DIR / 'outputs/figures/59_sparse_cnn_visual_examples_colab_large'
visual_report_dir = WORK_DIR / 'reports/sparse_cnn_visual_examples_colab_large'

cmd = [
    sys.executable, 'experiments/69_generate_sparse_cnn_visual_examples.py',
    '--model', str(model_path),
    '--eval-rows', str(out_dir / 'sparse_cnn_eval_rows.csv'),
    '--density', '0.03',
    '--examples-per-pattern', '1',
    '--patterns', 'Edge-Ring', 'Loc', 'Scratch', 'Donut', 'Random',
    '--out-dir', str(visual_dir),
    '--report-dir', str(visual_report_dir),
]
print('Running:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('visual_dir:', visual_dir)
print('visual_report_dir:', visual_report_dir)

## 9. Save Results Back To Google Drive

In [ ]:
import shutil
from pathlib import Path

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

items_to_copy = [
    out_dir,
    fig_dir,
    visual_dir,
    visual_report_dir,
    model_path,
]

for item in items_to_copy:
    src = Path(item)
    dst = RESULTS_DIR / src.name
    if dst.exists():
        if dst.is_dir():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print('copied', src, '->', dst)

print('Done. Results saved to:', RESULTS_DIR)

## 10. Quick Result Preview

In [ ]:
import pandas as pd

summary = pd.read_csv(out_dir / 'sparse_cnn_eval_summary.csv')
display(summary)

pattern_summary = pd.read_csv(out_dir / 'sparse_cnn_eval_pattern_summary.csv')
display(pattern_summary.sort_values('mean_average_precision').head(20))